# 02 — Classifier Training & Inference

This notebook demonstrates how to load and run the **MobileNetV3** 3-class
document classifier used in the MedVault OCR pipeline.

The classifier (`backend/classifier/classifier.py`) uses an ensemble approach:
- **CNN branch**: MobileNetV3-Large fine-tuned for 3 classes (TABLE, HANDWRITTEN, PRINTED_TEXT)
- **Heuristic branch**: Computer-vision feature scoring (line density, aspect ratio, etc.)
- **Fusion**: CNN prediction is used if confidence >= threshold; otherwise heuristics decide.

We will cover:
1. Loading the trained weights
2. Running inference on a single image
3. Inspecting the heuristic features
4. Training a new model from scratch (optional)

## Setup & Imports

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'backend'))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from backend.classifier.classifier import DocumentClassifier, ClassificationResult, CLASSES_3
from backend.classifier.heuristics import compute_features, score_features, FeatureVector

%matplotlib inline
print('Imports successful')
print(f'3-class labels: {CLASSES_3}')

## 1. Initialize the Classifier

The `DocumentClassifier` automatically loads trained weights from
`backend/weights/classifier_3class.pth` if they exist. If no weights are
found, it falls back to the heuristic-only mode.

In [ ]:
classifier = DocumentClassifier(
    confidence_threshold=0.70,
    line_density_threshold=0.0,
)

print(f'Device: {classifier.device}')
print(f'Model loaded: {classifier.model is not None}')
print(f'Num classes: {classifier.num_classes}')
print(f'Confidence threshold: {classifier.confidence_threshold}')

## 2. Load and Preprocess a Test Image

In [ ]:
IMAGE_PATH = str(ROOT / 'tests' / 'sample_images' / 'sample.png')

if not Path(IMAGE_PATH).exists():
    print('Sample not found; creating synthetic image...')
    synthetic = np.ones((800, 600, 3), dtype=np.uint8) * 240
    cv2.putText(synthetic, 'LAB REPORT', (150, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 2)
    cv2.putText(synthetic, 'ALT  45  U/L', (150, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 2)
    cv2.putText(synthetic, 'AST  32  U/L', (150, 260), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 2)
    IMAGE_PATH = str(ROOT / 'notebooks' / '_sample_synthetic.png')
    cv2.imwrite(IMAGE_PATH, synthetic)

image = cv2.imread(IMAGE_PATH)
print(f'Image shape: {image.shape}')

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title('Input Image')
plt.axis('off')
plt.show()

## 3. Run Classification

The `classify` method returns a `ClassificationResult` with:
- `doc_class`: The predicted class (TABLE / HANDWRITTEN / PRINTED_TEXT)
- `confidence`: CNN confidence score (0.0 to 1.0)
- `fallback_triggered`: True if heuristics were used instead of CNN
- `cnn_prediction` and `heuristic_prediction`: Individual branch outputs

In [ ]:
result = classifier.classify(image)

print('=== Classification Result ===')
print(f'Class:      {result.doc_class}')
print(f'Confidence: {result.confidence:.4f}')
print(f'Fallback:   {result.fallback_triggered}')
print(f'CNN pred:   {result.cnn_prediction}')
print(f'Heuristic:  {result.heuristic_prediction}')
print()
print('Serialized (wire format):')
print(result.to_dict())

## 4. Inspect Heuristic Features

The heuristic branch computes a `FeatureVector` from the image using
computer-vision techniques. Let's inspect these features to understand what
the heuristic branch sees.

In [ ]:
features = compute_features(image)

print('=== Extracted Features ===')
for field in features.__dataclass_fields__:
    value = getattr(features, field)
    print(f'  {field}: {value}')

print()
scores = score_features(features)
print('=== Heuristic Scores ===')
for cls, score in scores.items():
    print(f'  {cls}: {score:.4f}')

best_class = max(scores, key=scores.get)
print(f'\nHeuristic prediction: {best_class}')

## 5. Visualize the Classification Decision

Let's visualize the CNN confidence and heuristic scores side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heuristic scores bar chart
classes = list(scores.keys())
values = [scores[c] for c in classes]
colors = ['#2196F3' if c != best_class else '#4CAF50' for c in classes]
axes[0].bar(classes, values, color=colors)
axes[0].set_title('Heuristic Scores')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, max(values) * 1.2 if max(values) > 0 else 1)

# Final result
axes[1].barh(['Predicted Class'], [1], color='#4CAF50')
axes[1].text(0.5, 0, result.doc_class, ha='center', va='bottom', fontsize=14, fontweight='bold')
axes[1].set_title(f'Final: {result.doc_class} (conf={result.confidence:.2f})')
axes[1].set_xlim(0, 1)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 6. Training a New Model (Optional)

If you want to train the MobileNetV3 classifier from scratch or fine-tune
on new data, use the training script in `scripts/train_classifier.py`.

```bash
python scripts/train_classifier.py --data-dir /path/to/dataset --output backend/weights/classifier_3class.pth
```

The training script expects a directory structure:
```
dataset/
  TABLE/
    img1.png
    ...
  HANDWRITTEN/
    img1.png
    ...
  PRINTED_TEXT/
    img1.png
    ...
```

Below is a minimal training loop for reference:

In [ ]:
# This cell is for reference only -- uncomment and adapt to train.

# import torch
# import torch.nn as nn
# from torch.utils.data import DataLoader
# from torchvision.models import mobilenet_v3_large
#
# # Build model with 3 output classes
# model = mobilenet_v3_large(weights='IMAGENET1K_V1')
# model.classifier[3] = nn.Linear(model.classifier[3].in_features, 3)
# model = model.to(classifier.device)
#
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
#
# # Training loop (pseudo-code)
# # for epoch in range(num_epochs):
# #     for images, labels in train_loader:
# #         outputs = model(images)
# #         loss = criterion(outputs, labels)
# #         optimizer.zero_grad()
# #         loss.backward()
# #         optimizer.step()
#
# # Save weights
# # torch.save(model.state_dict(), str(ROOT / 'backend' / 'weights' / 'classifier_3class.pth'))

print('Training cell -- uncomment above to train a new model.')

## Summary

In this notebook we:
1. Loaded the MobileNetV3 3-class classifier with trained weights.
2. Ran inference on a sample image.
3. Inspected the heuristic feature vector and scores.
4. Visualized the classification decision.
5. Reviewed the training script entry point.

The classifier routes each document to one of three OCR engines:

| Class | OCR Engine |
|---|---|
| TABLE | PaddleOCR PP-Structure |
| HANDWRITTEN | Qwen2.5-VL |
| PRINTED_TEXT | PaddleOCR |